In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pathlib

import catboost as cb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from loguru import logger
from sklearn import metrics, pipeline

import highres_ta as ta

# Data loading and preprocessing

In [3]:
COORD_COLUMNS = [
    "expocode",
    "time",
    "lat",
    "lon",
    "depth",
]
TARGET_NAME = "talk"
FEATURE_NAMES = [
    "salinity",
    "temperature",
    "ssh_adt",
    "nitrate",
    "phosphate",
    "silicate",
    "bottomdepth",
]
QC_COLS = ["talkqc", "talkf"]
REQUIRED_COLUMNS = list(set(COORD_COLUMNS + FEATURE_NAMES + [TARGET_NAME] + QC_COLS))
NONAN_SUBSET = [TARGET_NAME, "salinity", "temperature", "nitrate", "ssh_adt"]
LINEAR_FEATURES = ["salinity", "temperature"]
FEATURE_NAMES += [
    "ncoord_x",
    "ncoord_y",
    "ncoord_z"
]

In [10]:
data = (
    ta.load_data()[REQUIRED_COLUMNS]
    .set_index(COORD_COLUMNS, drop=False)
    .pipe(ta.drop_extreme_salinities, min=20, max=40)
    .pipe(ta.add_talk_adjustment, fname="/Users/luke/Downloads/glodapv2_adjustments_last_updated_on_2026_07_09.csv")
    .pipe(ta.drop_bad_quality_talk)
    .dropna(subset=NONAN_SUBSET)
    .drop_duplicates(subset=COORD_COLUMNS, keep="first")
    .select_dtypes(include=[np.number])
    .pipe(ta.add_cyclical_dayofyear)
    .pipe(ta.add_spherical_coords)
    .loc[:, FEATURE_NAMES + [TARGET_NAME]]
)

2026-08-17 16:19:13.318 | DEBUG    | highres_ta.dataio:load_data:18 - Loading 40 .pq files from ../data/training
2026-08-17 16:19:13.473 | DEBUG    | highres_ta.target_filtering:drop_bad_quality_talk:49 - TA values with large adjustments (<= 6.0 mol/kg): 2365
2026-08-17 16:19:13.473 | DEBUG    | highres_ta.target_filtering:drop_bad_quality_talk:52 - TA values without good flags (!= 2): 3046
2026-08-17 16:19:13.473 | INFO     | highres_ta.target_filtering:drop_bad_quality_talk:53 - Number of rows filtered due to large adjustments and bad flags: 5319 of 41534 (13%)


## Train test split

In [29]:
train_idx, test_idx = ta.make_train_test_folds(data, n_splits=6)[0]
test = data.iloc[test_idx]
train = data.iloc[train_idx]
train_folds = ta.make_train_test_folds(train, shuffle=False)

2026-08-17 16:36:27.957 | DEBUG    | highres_ta.train_test_split:make_salinity_bins:48 - Using the following bin edges for salinity: [20.1909  32.8038  34.05706 34.819   35.535   39.231  ]
2026-08-17 16:36:27.961 | DEBUG    | highres_ta.train_test_split:stratified_group_folds:96 - Making train-test splits stratified by salinity_bin and grouped by expocode
2026-08-17 16:36:28.032 | DEBUG    | highres_ta.train_test_split:make_salinity_bins:48 - Using the following bin edges for salinity: [20.1909  32.804   34.05746 34.81948 35.535   37.584  ]
2026-08-17 16:36:28.034 | DEBUG    | highres_ta.train_test_split:stratified_group_folds:96 - Making train-test splits stratified by salinity_bin and grouped by expocode


# Catboost Tuning

## Hyperparameter tuning with nested cross-validation

Optuna tunes CatBoost on the outer training split only. Each inner fold fits its own linear baseline before training CatBoost on residuals, preventing validation-fold leakage. The objective is mean validation RMSE of the combined linear + boosted prediction.

In [12]:
RANDOM_SEED = 42
N_TRIALS = 50
NUM_THREADS = 1
MAX_ITERATIONS = 1000
EARLY_STOPPING_ROUNDS = 50
ALWAYS_IGNORED_FEATURES = sorted(set(COORD_COLUMNS).intersection(FEATURE_NAMES))

fixed_catboost_params = {
    "loss_function": "RMSEWithUncertainty",
    "iterations": 2000,
    "random_seed": RANDOM_SEED,
    "ignored_features": ALWAYS_IGNORED_FEATURES,
    "allow_writing_files": False,
    "thread_count": NUM_THREADS,
    "verbose": False,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
}

fixed_catboost_params

{'loss_function': 'RMSEWithUncertainty',
 'iterations': 2000,
 'random_seed': 42,
 'ignored_features': [],
 'allow_writing_files': False,
 'thread_count': 1,
 'verbose': False,
 'early_stopping_rounds': 50}

In [38]:
def objective(trial: optuna.Trial) -> float:
    trial_params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.00, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 100.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
        "depth": trial.suggest_int("depth", 4, 12),
        "rsm": trial.suggest_float("rsm", 0.0, 1.0),
    }

    fold_rmse = []
    fold_r2score = []
    fold_best_iterations = []

    boost_kwargs = fixed_catboost_params | trial_params

    for fold_number, (fold_train_idx, fold_valid_idx) in enumerate(train_folds):
        
        fold_model = ta.CatBoostResidualRegressor(
            linear_features=LINEAR_FEATURES,
            feature_names=FEATURE_NAMES,
            **boost_kwargs
        )

        x_train = train.iloc[fold_train_idx].loc[:, FEATURE_NAMES]
        y_train = train.iloc[fold_train_idx].loc[:, TARGET_NAME]
        x_valid = train.iloc[fold_valid_idx].loc[:, FEATURE_NAMES]
        y_valid = train.iloc[fold_valid_idx].loc[:, TARGET_NAME]

        fold_model.fit(
            x_train,
            y_train,
            eval_set=(x_valid, y_valid)
        )

        
        rmse = fold_model.score(x_valid, y_valid, metric="root_mean_squared_error")
        r2score = fold_model.score(x_valid, y_valid, metric="r2_score")

        fold_rmse.append(float(rmse))
        fold_r2score.append(float(r2score))
        fold_best_iterations.append(fold_model.boosting_model_.get_best_iteration() + 1)

        trial.report(float(np.median(fold_rmse)), step=fold_number)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr("fold_rmse", fold_rmse)
    trial.set_user_attr("fold_r2score", fold_r2score)
    trial.set_user_attr("fold_best_iterations", fold_best_iterations)
    trial.set_user_attr("refit_iterations", int(np.median(fold_best_iterations)))
    
    return float(np.median(fold_rmse))

In [39]:
sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
pruner = optuna.pruners.NopPruner()
study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    storage="sqlite:///catboost_uncertainty_cv.db",
    study_name=f"catboost_uncertainty_{fixed_catboost_params['loss_function']}_ignored_features_v04",
    load_if_exists=True
)
study.optimize(objective, n_trials=400, n_jobs=4, show_progress_bar=True)

[I 2026-08-17 16:38:14,650] Using an existing study with name 'catboost_uncertainty_RMSEWithUncertainty_ignored_features_v04' instead of creating a new one.


  0%|          | 0/400 [00:00<?, ?it/s]

[I 2026-08-17 16:38:15,160] Trial 408 pruned. 
[I 2026-08-17 16:38:15,208] Trial 411 pruned. 
[I 2026-08-17 16:38:15,247] Trial 410 pruned. 
[I 2026-08-17 16:38:15,528] Trial 409 pruned. 
[I 2026-08-17 16:38:15,722] Trial 414 pruned. 
[I 2026-08-17 16:38:16,430] Trial 415 pruned. 
[I 2026-08-17 16:38:16,489] Trial 413 pruned. 
[I 2026-08-17 16:38:16,736] Trial 412 pruned. 
[I 2026-08-17 16:38:16,796] Trial 416 pruned. 
[I 2026-08-17 16:38:16,995] Trial 418 pruned. 
[I 2026-08-17 16:38:17,156] Trial 417 pruned. 
[I 2026-08-17 16:38:17,185] Trial 419 pruned. 
[I 2026-08-17 16:38:17,246] Trial 420 pruned. 
[I 2026-08-17 16:38:17,302] Trial 422 pruned. 
[I 2026-08-17 16:38:17,367] Trial 424 pruned. 
[I 2026-08-17 16:38:17,542] Trial 421 pruned. 
[I 2026-08-17 16:38:17,834] Trial 427 pruned. 
[I 2026-08-17 16:38:17,841] Trial 425 pruned. 
[I 2026-08-17 16:38:17,847] Trial 423 pruned. 
[I 2026-08-17 16:38:18,029] Trial 426 pruned. 
[I 2026-08-17 16:38:18,756] Trial 428 pruned. 
[I 2026-08-17

In [15]:
from optuna import visualization

cv_results = (
    study.trials_dataframe(
        attrs=("number", "value", "params", "user_attrs", "state")
    )
    .sort_values("value", na_position="last")
    .reset_index(drop=True)
)

print(f"Best mean CV RMSE: {study.best_value:.3f}")
print(f"Refit iterations: {study.best_trial.user_attrs['refit_iterations']}")
cv_results.head(10)
visualization.plot_parallel_coordinate(study)

Best mean CV RMSE: 18.123
Refit iterations: 75


## Refit and held-out evaluation

The selected hyperparameters are refit on all outer-training observations. The held-out test split is used once for the final metrics and was not used by Optuna or early stopping.

In [30]:
best_trial_params = {
    name: value
    for name, value in study.best_params.items()
    if name not in {"ignored_feature_1", "ignored_feature_2"}
}
best_params = {
    **(fixed_catboost_params | best_trial_params),
    "iterations": study.best_trial.user_attrs["refit_iterations"],
}
boosted_trees_model = ta.CatBoostResidualRegressor(LINEAR_FEATURES, FEATURE_NAMES, polynomial_degree=1, **best_params)
boosted_trees_model.fit(train[FEATURE_NAMES], train[TARGET_NAME])

,linear_features,"['salinity', 'temperature']"
,feature_names,"['salinity', 'temperature', ...]"
,polynomial_degree,1


In [31]:
subset = test
test_prediction = boosted_trees_model.predict(subset)

test_metrics = pd.Series(
    {
        "rmse": metrics.root_mean_squared_error(
            subset[TARGET_NAME], test_prediction[:, 0]
        ),
        "mae": metrics.mean_absolute_error(
            subset[TARGET_NAME], test_prediction[:, 0]
        ),
        "r2": metrics.r2_score(subset[TARGET_NAME], test_prediction[:, 0]),
    },
    name="held_out_test",
)
test_metrics

rmse    17.504499
mae      8.941894
r2       0.968869
Name: held_out_test, dtype: float64

In [ ]:
feature_importance = pd.Series(
    boosted_trees_model.boosting_model_.feature_importances_,
    index=boosted_trees_model.boosting_model_.feature_names_).sort_values()

# feature_importance.plot.barh(figsize=(8, 6), title="Feature Importance")